# 05 — Inspect imagery, QA, and diagnostics

This notebook turns the output checks from `Raster_processing.ipynb` into a repeatable QA step. It can render the package QA panel, print its machine-readable metrics, and display an ENVI band or RGB composite for direct spatial inspection.

## 1. Configure one completed flightline

Choose the corrected ENVI image produced by the same flightline run. The plotting band indices are diagnostic display choices; verify them against the ENVI wavelength metadata before interpreting color.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from spectral import open_image

from spectralbridge.qa_plots import render_flightline_panel

RUN = False
base_folder = Path("outputs/neon_notebook")
flight_stem = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
flight_dir = base_folder / flight_stem
corrected_candidates = sorted(flight_dir.glob("*brdfandtopo_corrected_envi.img")) if flight_dir.exists() else []
corrected_img_path = corrected_candidates[0] if corrected_candidates else None

## 2. Reuse the development plotting helpers

These are the same lightweight ENVI inspection patterns used in the root research notebook, with `Path` handling and zero-range protection added. They read only the selected display bands.

In [ ]:
def plot_envi_band(img_path: Path, band_index: int = 0, cmap: str = "gray") -> None:
    """Display one zero-based ENVI band for spatial inspection."""
    image = open_image(str(img_path.with_suffix(".hdr")))
    data = np.asarray(image[:, :, band_index], dtype=float).squeeze()
    plt.figure(figsize=(8, 8))
    plt.imshow(data, cmap=cmap)
    plt.title(f"ENVI band {band_index}")
    plt.colorbar(label="Reflectance")
    plt.axis("off")
    plt.show()


def plot_envi_rgb(
    img_path: Path,
    rgb_bands: tuple[int, int, int] = (29, 19, 9),
    stretch: tuple[float, float] = (2, 98),
) -> None:
    """Display three selected bands with a per-channel percentile stretch."""
    image = open_image(str(img_path.with_suffix(".hdr")))
    rgb = np.asarray(image[:, :, list(rgb_bands)], dtype=float)
    low, high = np.percentile(rgb, stretch, axis=(0, 1))
    span = np.where(high > low, high - low, 1.0)
    rgb = np.clip((rgb - low) / span, 0, 1)
    plt.figure(figsize=(8, 8))
    plt.imshow(rgb)
    plt.title(f"RGB composite (bands {rgb_bands})")
    plt.axis("off")
    plt.show()

## 3. Render and inspect QA

The package panel summarizes the full flightline. The direct ENVI views help confirm that those aggregate diagnostics are consistent with visible spatial structure.

In [ ]:
existing_qa = sorted(flight_dir.glob("*_qa.*")) if flight_dir.exists() else []
print(f"Corrected image: {corrected_img_path}")
print(f"Existing QA artifacts: {[path.name for path in existing_qa]}")

if RUN and corrected_img_path is not None:
    qa_png, metrics = render_flightline_panel(flight_dir, quick=True, save_json=True)
    print(f"QA figure: {qa_png}")
    print(json.dumps(metrics, indent=2, default=str)[:5000])
    plot_envi_band(corrected_img_path, band_index=1)
    plot_envi_rgb(corrected_img_path)
elif corrected_img_path is None:
    print("No corrected ENVI image found. Run or resume notebook 00.")
else:
    print("Set RUN = True to render QA and display the selected ENVI image.")

## 4. What to review

Look for NoData dominance, striping, discontinuities, implausible reflectance ranges, and correction artifacts. Record both the QA JSON and the figure. A plot that looks plausible is supporting evidence, not a substitute for the quantitative validation pages.